# 팀 실습 — 공개 정리본
교육 과정의 제공 골격을 바탕으로 수행한 실습입니다. 개인/팀 범위를 폴더로 구분했습니다.
공개 정리 시 환경 연결 셀·데이터 미리보기·내장 교안 이미지를 제외했습니다. 학습 설정과 모델 로직은 원본을 유지합니다.
개인 노트북의 평가 텍스트는 당시 저장 결과이며 나머지 실행 출력은 정리했습니다.


In [ ]:
import os


# **Step1. 상품 리뷰 감성 분류**

**미션 : 리뷰마다 긍/부정 감성분석하기**
1) 목표 : 리뷰를 긍, 부정으로 분류하는 모델 생성
사전학습 모델을 파인 튜닝하여 모델 성능 높이기
2) 절차
    * 데이터 전처리
        - 긍/부정 분류를 위한 데이터셋 구성
        - 데이터 불균형 문제 해결하기
    * 사전학습 모델 파인 튜닝 : klue/bert-base 사용 권장
3) 최소 4가지 이상의 실험을 구성하고 수행하고 성능 비교
    * 학습 + 검증 데이터 : 10만 건 중 일부를 샘플링해서 수행.
    * 모델링 : 일반 파인튜닝, LoRA 파인튜닝 적용
    * 하이퍼파라미터 조정


## 1.환경준비

### (1) 라이브러리 설치 및 로딩

In [ ]:
!pip install datasets peft accelerate -q

* 설치 후 세션 다시 시작

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder

from peft import get_peft_model, LoraConfig, TaskType

from warnings import filterwarnings
FutureWarning
filterwarnings('ignore')

* 함수 생성

In [ ]:
# 검증셋 평가 함수
def evaluate(val_ds, model, device, tokenizer):
    # 입력 데이터셋 토크나이징 (attention_mask 포함)
    inputs = tokenizer(list(val_ds['text']), return_tensors="pt", padding=True,
                       truncation=True, max_length=128
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}  # 입력 텐서를 동일한 디바이스로 이동

    # 모델을 지정된 디바이스로 이동
    model = model.to(device)

    with torch.no_grad():  # 평가 과정에서 기울기 계산 비활성화
        outputs = model(**inputs)  # attention_mask를 포함해 입력

    # 예측 및 확률 계산
    probabilities = outputs.logits.softmax(dim=1)

    # probabilities가 GPU에 있을 경우에만 CPU로 이동
    if probabilities.is_cuda:
        probabilities = probabilities.cpu().detach().numpy()
    else:
        probabilities = probabilities.detach().numpy()

    pred = np.argmax(probabilities, axis=1)

    # GPU 메모리에서 필요 없는 텐서 제거 및 캐시 정리
    del inputs
    torch.cuda.empty_cache()

    return pred, probabilities

In [ ]:
def predict(text, model, tokenizer):
    # 모델을 CPU로 이동
    model = model.to("cpu")

    # 입력 문장 토크나이징 → CPU 텐서로 생성
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    # 모델 예측 (no_grad로 메모리 절약)
    with torch.no_grad():
        outputs = model(**inputs)

    # 확률 계산
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=1)

    # 예측 클래스
    pred = torch.argmax(probabilities, dim=-1).item()

    return pred, probabilities


### (2) GPU 설정
* GPU 사용을 위한 설정

In [ ]:
# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2.데이터 준비① : 데이터 구조 만들기
* 아래와 같은 형태로 데이터 셋을 준비합니다.
    * 하나의 리뷰 문장에 하나의 긍/부정 값 지정
    * 리뷰별 label의 평균 값 >= 0.5 ➔ 긍정(1)

* 데이터 불균형을 확인하고 조치합니다.

### (1) 데이터 불러오기
* 제공 받은 'data.csv' 파일을 불러 옵니다.

In [ ]:
# 공개본: 환경 연결 셀은 생략. 데이터 경로는 DATA_PATH로 지정합니다.


In [ ]:
data = pd.read_csv(os.environ.get("DATA_PATH", "../../data/data.csv"))
data

### (2) 데이터 구조 만들기

* 다음과 같은 형태로 리뷰별 데이터 구조를 만듭니다.

In [ ]:
data.head()

In [ ]:
#한 리뷰에서 1이 더 많으면 긍정(그룹바이 해서 label의 평균값이 0.5보다 크면 1, 아니면 0)

data_review = data.groupby('text', as_index=False)['label'].mean()

In [ ]:
data_review['label'] = np.where(data_review['label'] > 0.5, 1, 0)

### (3) 데이터 불균형 해결하기

* label의 분포 확인하기

In [ ]:
# sns.countplot()
sns.countplot(data = data_review, x = 'label')

* label이 0과 1인 인 경우 각각 샘플링 2000건씩
    * 더 많이 할 수도 있지만 학습 시간을 고려


In [ ]:
# 데이터 샘플링
df_1 = data_review.loc[data_review['label']==1].sample(2000)
df_0 = data_review.loc[data_review['label']==0].sample(2000)
data_0 = pd.concat([df_1,df_0], axis = 0)

In [ ]:
# 데이터 분포 다시 확인
sns.countplot(data = data_0, x = 'label')

## 3.데이터 준비② : 데이터 전처리

### (1) Dataset 만들기 : train, val
* train_test_split 함수 이용

In [ ]:
from sklearn.model_selection import train_test_split

train, val = train_test_split(data_0, test_size=0.2, random_state=42)

* 텐서 데이터셋 만들기

In [ ]:
from datasets import Dataset

train_ts = Dataset.from_pandas(train)
val_ts = Dataset.from_pandas(val)


In [ ]:
train_ts[:3]

### (2) 토크나이징
* 모델 : "klue/bert-base"

In [ ]:
# 모델과 토크나이저 불러오기
from transformers import AutoTokenizer

model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 토큰화 함수 생성 및 작업
def preprocess_function(data):
    return tokenizer(data['text'], truncation=True, padding=True)

train_ts = train_ts.map(preprocess_function, batched=True)
val_ts = val_ts.map(preprocess_function, batched=True)

In [ ]:
train_ts[0]

## 4.Fine-Tuning ①
* 일반 파인 튜닝 수행

### (1) 사전학습 모델 준비

In [ ]:
# 모델 로드
n = 2
model = AutoModelForSequenceClassification.from_pretrained(model_name,
                                                           num_labels = n).to(device)

### (2) 학습 설정

In [ ]:
# TrainingArguments 설정
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-5,               # 작은 학습률
    per_device_train_batch_size = 32,   # 학습 배치 사이즈
    per_device_eval_batch_size = 32,
    num_train_epochs = 5,               # 에폭 수
    weight_decay = 0.02,                # weight decay
    load_best_model_at_end = True,      # earlystopping 사용하기 위해 필요
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

In [ ]:
# Trainer 설정
trainer = Trainer(
    model=model,                         # 학습할 모델
    args=training_args,                  # TrainingArguments
    train_dataset = train_ts,
    eval_dataset = val_ts,
    tokenizer = tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=3)], # 조기 종료
)

### (3) 학습

In [ ]:
trainer.train()

### (4) 모델 검증평가

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

In [ ]:
text = "촉촉해서 사용감이 너무 좋아요"
predicted_class, probabilities = predict(text, model, tokenizer)

print(f"예측된 클래스: {predicted_class}")
print(f"클래스별 확률: {probabilities}")

In [ ]:
pred, prob = evaluate(val_ts, model, device, tokenizer)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix(val_ts['label'], pred))
disp.plot()
plt.show()

print(classification_report(val_ts['label'], pred))

## 5.Fine-Tuning②
* LoRA(Low-Rank Adaptation)

### (1) 사전학습 모델 준비
* 위에서 적용한 내용

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
# 모델 로드

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=n)

### (2) LoRA 구성

In [ ]:
# 1) 선형층 이름 스캔
linear_names = []
for name, module in base_model.named_modules():
    if isinstance(module, torch.nn.Linear):
        linear_names.append(name.split(".")[-1])  # 마지막 토큰만 추출 (ex. q_lin)

# 2) 모델군별 후보 세트
candidates_sets = [
    # DistilBERT 계열
    ["q_lin", "v_lin"],                     # 최소
    ["q_lin", "k_lin", "v_lin", "out_lin"], # 풀 어텐션
    # BERT/Roberta/DeBERTa/KoBERT 계열
    ["query", "value"],
    ["query", "key", "value"],
]

# 3) 실제로 존재하는 이름만 남겨 타겟 확정
target = None
for cand in candidates_sets:
    exist = [n for n in cand if n in linear_names]
    if len(exist) >= 2:   # 최소 쿼리/밸류 두 개 이상이면 채택
        target = exist
        break

if target is None:
    raise ValueError(f"LoRA를 꽂을 선형층을 찾지 못했습니다. "
                     f"발견된 선형층 끝이름 예시: {sorted(set(linear_names))}")

print("LoRA target_modules =", target)


In [ ]:
# LoRA 구성 정의
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # 시퀀스 분류
    r=16,                        # 랭크 값 (작을수록 파라미터 적음)
    lora_alpha=64,               # 너무 작으면 학습 효과 약하고, 너무 크면 발산 위험
    lora_dropout=0.1,
    target_modules=["query", "value"],  # DistilBERT->klue/bert-base에서 사용되는 attention 레이어 이름
    bias="none"                 # bias는 학습하지 않음
)

# LoRA 적용 (일부 레이어만 학습)
model = get_peft_model(base_model, lora_config)

### (3) 학습

In [ ]:
# TrainingArguments (기존 Trainer 그대로 사용 가능)
training_args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-4,               # LoRA는 조금 큰 학습률
    per_device_train_batch_size = 32,   # 학습 배치 사이즈
    per_device_eval_batch_size = 32,
    num_train_epochs = 10,               # 에폭 수
    weight_decay = 0.02,                # weight decay
    load_best_model_at_end = True,      # earlystopping 사용하기 위해 필요
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

# Trainer 설정 (compute_metrics, callbacks 등 필요 시 추가)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ts,
    eval_dataset=val_ts,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# 학습 시작
trainer.train()

### (4) 모델 검증평가

In [ ]:
text = "촉촉해서 사용감이 너무 좋아요"
predicted_class, probabilities = predict(text, model, tokenizer)

print(f"예측된 클래스: {predicted_class}")
print(f"클래스별 확률: {probabilities}")

In [ ]:
pred, prob = evaluate(val_ts, model, device, tokenizer)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix(val_ts['label'], pred))
disp.plot()
plt.show()

print(classification_report(val_ts['label'], pred))